In [2]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'  # Set torch cache directory

In [3]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import networkx as nx
from pyvis.network import Network

In [4]:
# ============================================================
# 1. CONFIG
# ============================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
NUM_CLASSES = 7

In [5]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [11]:
dataset_artpainting = datasets.ImageFolder("/home/aban/circuit_tracing/pacs/pacs_data/pacs_data/sketch", transform=transform)

generator = torch.Generator().manual_seed(42)
train_size = int(0.8 * len(dataset_artpainting))
test_size = len(dataset_artpainting) - train_size

train_dataset_artpainting, test_dataset_artpainting = random_split(dataset_artpainting, [train_size, test_size], generator=generator)

train_loader_artpainting = DataLoader(train_dataset_artpainting, batch_size=BATCH_SIZE, shuffle=True, num_workers = 4)
test_loader_artpainting = DataLoader(test_dataset_artpainting, batch_size=BATCH_SIZE, num_workers = 4)

In [12]:
print("Number of samples:", len(dataset_artpainting))
print("Number of classes:", len(dataset_artpainting.classes))
print("Classes:", dataset_artpainting.classes)

Number of samples: 3929
Number of classes: 7
Classes: ['dog', 'elephant', 'giraffe', 'guitar', 'horse', 'house', 'person']


In [13]:
# -----------------------------
# MODEL
# -----------------------------
class MLPAdapter(nn.Module):
    def __init__(self, channels):
        super().__init__()
        hidden = max(channels // 4, 16)
        self.mlp = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.ReLU(),
            nn.Linear(hidden, channels)
        )

    def forward(self, x):
        B,C,H,W = x.shape
        x_flat = x.permute(0,2,3,1).reshape(-1,C)
        out = self.mlp(x_flat)
        out = out.view(B,H,W,C).permute(0,3,1,2)
        return x + out

In [14]:
class VGG19_Adapter(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        base = models.vgg19(weights="IMAGENET1K_V1")

        for p in base.features.parameters():
            p.requires_grad = False

        self.features = base.features

        # SAFE adapter naming (no dots)
        self.adapter_layers = [28, 30, 32]
        self.adapters = nn.ModuleDict({
            str(i): MLPAdapter(512) for i in self.adapter_layers
        })

        self.pool = base.avgpool

        self.classifier = nn.Sequential(
            nn.Linear(512*7*7, 1024), nn.ReLU(),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        for name, layer in self.features._modules.items():
            x = layer(x)
            if name in self.adapters:
                x = self.adapters[name](x)
        x = self.pool(x)
        x = torch.flatten(x,1)
        return self.classifier(x)

In [15]:
model_vgg19_sketch = VGG19_Adapter(NUM_CLASSES).to(DEVICE)

In [16]:
# Load trained weights
checkpoint = torch.load(
    "/home/aban/circuit_tracing/circuits/outputs_pacs/best_model_sketch_vgg.pth",
    map_location=DEVICE
)

/tmp/ipykernel_2236440/4272596381.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(


In [17]:
# Handle checkpoints saved as either state_dict or dict
if "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
else:
    state_dict = checkpoint

# Remove "module." prefix if trained with DataParallel/DDP
state_dict = {
    k.replace("module.", ""): v
    for k, v in state_dict.items()
}

model_vgg19_sketch.load_state_dict(state_dict)

model_vgg19_sketch = model_vgg19_sketch.to(DEVICE)
model_vgg19_sketch.eval()

VGG19_Adapter(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): Conv2d(256, 256, kernel_size=(3, 3), stride=(1

In [18]:
# ============================================================
# 5. EVALUATION
# ============================================================
correct = 0
total = 0

class_correct = torch.zeros(NUM_CLASSES)
class_total = torch.zeros(NUM_CLASSES)

with torch.no_grad():

    for images, labels in test_loader_artpainting:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model_vgg19_sketch(images)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        # Class-wise accuracy
        for cls in range(NUM_CLASSES):
            mask = labels == cls

            class_total[cls] += mask.sum().item()
            class_correct[cls] += (
                (predictions == labels) & mask
            ).sum().item()


# ============================================================
# 6. RESULTS
# ============================================================
accuracy = 100.0 * correct / total

print("\n==============================")
print("Evaluation Results")
print("==============================")

print(f"Overall Accuracy: {accuracy:.2f}%")

print("\nClass-wise Accuracy:")

for cls in range(NUM_CLASSES):

    if class_total[cls] > 0:
        cls_acc = (
            100.0
            * class_correct[cls]
            / class_total[cls]
        )

        print(
            f"{dataset_artpainting.classes[cls]:20s}: "
            f"{cls_acc:.2f}% "
            f"({int(class_correct[cls])}/"
            f"{int(class_total[cls])})"
        )


Evaluation Results
Overall Accuracy: 86.90%

Class-wise Accuracy:
dog                 : 74.00% (111/150)
elephant            : 88.03% (125/142)
giraffe             : 89.57% (146/163)
guitar              : 99.22% (128/129)
horse               : 84.38% (135/160)
house               : 100.00% (13/13)
person              : 86.21% (25/29)


In [19]:
##############################################################################
# LOAD CIRCUITS
##############################################################################
import pickle
with open("/home/aban/circuit_tracing/circuits/outputs_pacs/circuits_vgg_sketch.pkl", "rb") as f:
    circuits = pickle.load(f)

In [20]:
##############################################################################
# HELPER FUNCTIONS
##############################################################################

def graph_to_layer_channels(graph):
    """
    Convert a NetworkX graph into

    {
        "layer1":[...],
        "layer2":[...],
        ...
    }
    """
    layer_channels = {}

    for node in graph.nodes():
        layer, ch = node.split("_ch")
        ch = int(ch)

        if layer not in layer_channels:
            layer_channels[layer] = []

        layer_channels[layer].append(ch)

    return layer_channels

In [21]:
def make_hook(channels):
    def hook(module, inp, out):
        out = out.clone()
        out[:, channels] = 0
        return out
    return hook

In [22]:
@torch.no_grad()
def class_accuracy(model, loader):

    correct = torch.zeros(NUM_CLASSES)
    total = torch.zeros(NUM_CLASSES)

    model.eval()

    for images, labels in loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        pred = model(images).argmax(1)

        for cls in range(NUM_CLASSES):

            mask = labels == cls

            correct[cls] += (pred[mask] == cls).sum().item()
            total[cls] += mask.sum().item()

    return correct / total

In [23]:
import copy
@torch.no_grad()
def evaluate_with_circuit(model, graph, loader):

    model_copy = copy.deepcopy(model)

    handles = []

    layer_channels = graph_to_layer_channels(graph)

    for layer, channels in layer_channels.items():

        h = model_copy.adapters[layer].register_forward_hook(
            make_hook(channels)
        )

        handles.append(h)

    acc = class_accuracy(model_copy, loader)

    for h in handles:
        h.remove()

    return acc

In [26]:
##############################################################################
# BASELINE
##############################################################################

baseline = class_accuracy(model_vgg19_sketch, test_loader_artpainting)

##############################################################################
# EVALUATE ALL CIRCUITS
##############################################################################

results = {}

for circuit_id, graph in circuits.items():

    print(f"Evaluating circuit {circuit_id}")

    results[circuit_id] = evaluate_with_circuit(
        model_vgg19_sketch,
        graph,
        test_loader_artpainting,
    )

##############################################################################
# PRINT DIFFERENCE MATRIX
##############################################################################

print("\nDifference Matrix\n")

matrix = torch.zeros(NUM_CLASSES, NUM_CLASSES)

for circuit in range(NUM_CLASSES):

    diff = results[circuit] - baseline
    matrix[circuit] = diff

print(matrix)

##############################################################################
# FIND SMALLEST DIFFERENCE FOR EACH CLASS
##############################################################################

print("\n")
print("=" * 70)
print("SMALLEST DIFFERENCE FOR EACH CLASS")
print("=" * 70)

smallest = []

for cls in range(NUM_CLASSES):

    best_circuit = None
    best_value = None

    for circuit in range(NUM_CLASSES):

        value = matrix[circuit, cls].item()

        if best_value is None or abs(value) < abs(best_value):
            best_value = value
            best_circuit = circuit

    smallest.append(
        {
            "class": cls,
            "circuit": best_circuit,
            "difference": best_value,
        }
    )

for item in smallest:
    print(
        f"Class {item['class']} | "
        f"Best Circuit {item['circuit']} | "
        f"Difference {item['difference']:+.6f}"
    )

##############################################################################
# OPTIONAL: LIST ONLY THE 7 VALUES
##############################################################################

smallest_values = [x["difference"] for x in smallest]

print("\nOnly the 7 smallest values (one per class):")
print(smallest_values)

Evaluating circuit 0
Evaluating circuit 1
Evaluating circuit 2
Evaluating circuit 3
Evaluating circuit 4
Evaluating circuit 5
Evaluating circuit 6

Difference Matrix

tensor([[-0.7467, -0.5563,  0.0429,  0.0155, -0.2438,  0.0000, -0.3793],
        [-0.7467, -0.7394,  0.0245,  0.0000, -0.1250,  0.0000, -0.3448],
        [-0.7467, -0.1690, -0.1104,  0.0078,  0.0687,  0.0000, -0.2414],
        [-0.5933, -0.0282, -0.0429,  0.0078,  0.0250,  0.0000, -0.1724],
        [-0.4333,  0.0352, -0.0613,  0.0155, -0.3688,  0.0000, -0.1724],
        [-0.6267,  0.0986, -0.0859, -0.0698,  0.0187, -1.0000, -0.4483],
        [-0.0200, -0.0704, -0.0123, -0.0078,  0.0062, -0.0769, -0.6897]])


SMALLEST DIFFERENCE FOR EACH CLASS
Class 0 | Best Circuit 6 | Difference -0.020000
Class 1 | Best Circuit 3 | Difference -0.028169
Class 2 | Best Circuit 6 | Difference -0.012270
Class 3 | Best Circuit 1 | Difference +0.000000
Class 4 | Best Circuit 6 | Difference +0.006250
Class 5 | Best Circuit 0 | Difference +0.000

In [27]:
print("=" * 60)
print("Circuit-wise Performance")
print("=" * 60)

for circuit_id, graph in circuits.items():

    acc = evaluate_with_circuit(model_vgg19_sketch, graph, test_loader_artpainting)

    base_acc = baseline[circuit_id].item()
    circuit_acc = acc[circuit_id].item()
    diff = circuit_acc - base_acc

    print(
        f"Circuit {circuit_id}: "
        f"Baseline = {base_acc:.4f} | "
        f"Circuit = {circuit_acc:.4f} | "
        f"Difference = {diff:+.4f}"
    )

Circuit-wise Performance
Circuit 0: Baseline = 0.7467 | Circuit = 0.0000 | Difference = -0.7467
Circuit 1: Baseline = 0.8803 | Circuit = 0.1761 | Difference = -0.7042
Circuit 2: Baseline = 0.8896 | Circuit = 0.7914 | Difference = -0.0982
Circuit 3: Baseline = 0.9845 | Circuit = 1.0000 | Difference = +0.0155
Circuit 4: Baseline = 0.8500 | Circuit = 0.5063 | Difference = -0.3438
Circuit 5: Baseline = 1.0000 | Circuit = 0.0000 | Difference = -1.0000
Circuit 6: Baseline = 0.8621 | Circuit = 0.1379 | Difference = -0.7241
